# TCRmeta smoke test

Exercises all four public functions end to end: `embed_repertoire`, `compute_css`, `plot_umap`, `energy_shift`, plus `build_reference`.

**Fixed to a specific real sample + a specific, explicitly-loaded reference map** (rather than relying on defaults/fallbacks),
so results here are directly comparable to `debug_plot_umap.ipynb` — same repertoire, same reference file, same
downsample/seed/grid parameters. If you point `SAMPLE_CSV`/`MODEL_PATH` at the same files in both notebooks, the
figures/numbers should match exactly.

In [ ]:
import os

# --- EDIT ME: pick ONE of these two, or leave both unset if the Hub repo
# already exists and is public/you're logged in via `huggingface-cli login`.

# os.environ["TCRMETA_HF_REPO"] = "yourname/tcrmeta-weights"
# os.environ["TCRMETA_WEIGHTS_DIR"] = "/path/to/local/weights/folder"

import time
import numpy as np
import pandas as pd

import tcrmeta as tm

print("tcrmeta version:", tm.__version__)

import torch
print("CUDA available:", torch.cuda.is_available())

## Parameters

Set these to real files. `REPERTOIRE_1_PATH`/`REPERTOIRE_2_PATH` must be real, existing CSVs — this notebook no longer
silently falls back to synthetic data, specifically to avoid the confusing situation where a typo'd path produces a
plausible-looking-but-meaningless plot instead of an error.

In [ ]:
REPERTOIRE_1_PATH = "/mnt/isilon/boli_lab/globus/project/mingyao/tcr_database/HIV/csv/015V08002889_CFAR.csv"
REPERTOIRE_2_PATH = "/path/to/your/repertoire_2.csv"  # EDIT ME to a second real sample for energy_shift

MODEL_PATH = (
    "/mnt/isilon/boli_lab/globus/project/mingyao/DeepCluster/distribution_shift/"
    "Tonon/reference_map/density_map_output/density_map_output_pca30_10000_thres/"
    "vgene_density_models.pkl"
)

V_GENE = "TRBV27"
DOWNSAMPLE = 10000
SEED = 42
GRID_N = 300

# Common Adaptive/AIRR column name variants -> tcrmeta's required exact names.
COL_MAP = {
    "aminoAcid": "cdr3aa",
    "vGeneName": "v_gene",
    "count (templates/reads)": "count",
    "templates": "count",
    "junction_aa": "cdr3aa",
    "v_call": "v_gene",
    "duplicate_count": "count",
}


def load_repertoire(path):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{path} does not exist. Set this to a real repertoire CSV \u2014 "
            "this notebook intentionally does not fall back to synthetic data."
        )
    df_raw = pd.read_csv(path)
    df = df_raw.rename(columns={k: v for k, v in COL_MAP.items() if k in df_raw.columns})
    if "count" not in df.columns and "productive_frequency" in df.columns:
        df["count"] = (df["productive_frequency"].astype(float) * 1_000_000).round().astype(int)
    df = df[["cdr3aa", "v_gene", "count"]].copy()
    print(f"Loaded {path}: {len(df)} rows, {df['v_gene'].nunique()} unique V genes")
    return df


df1 = load_repertoire(REPERTOIRE_1_PATH)
df1.head()

## Load the reference map explicitly

Passed explicitly to every call below (`reference=reference`) instead of relying on the default/auto-downloaded
reference, so we're guaranteed to be using the exact same reference file as `debug_plot_umap.ipynb`.

In [ ]:
reference = tm.load_reference(MODEL_PATH)
print(f"Loaded reference: {len(reference)} V gene models, {V_GENE} present: {V_GENE in reference}")

## 1. Task 1 — `embed_repertoire`

In [ ]:
t0 = time.time()
embeddings = tm.embed_repertoire(df1, device="auto")
print(f"Embedded {len(embeddings)} unique clones in {time.time() - t0:.1f}s")

example_key = next(iter(embeddings))
print("Example clone key:", example_key)
print("Embedding shape:", embeddings[example_key].shape)

## 2. Task 2 — `compute_css` (explicit reference + downsample + seed)

In [ ]:
t0 = time.time()
per_gene_css, whole_css = tm.compute_css(
    df1,
    reference=reference,
    embeddings=embeddings,
    downsample=DOWNSAMPLE,
    seed=SEED,
    device="auto",
)
print(f"compute_css took {time.time() - t0:.1f}s")

print(f"\n{V_GENE} row:")
display(per_gene_css.loc[per_gene_css["v_gene"] == V_GENE])

print("\nWhole-repertoire CSS:")
display(whole_css)

## 3. Task 3 — `plot_umap` (same V_GENE, same explicit reference/embeddings/downsample/seed/grid_n as debug_plot_umap.ipynb)

In [ ]:
fig, coords_df = tm.plot_umap(
    df1,
    reference=reference,
    v_gene=V_GENE,
    embeddings=embeddings,
    downsample=DOWNSAMPLE,
    seed=SEED,
    grid_n=GRID_N,
    device="auto",
)
print(f"[{V_GENE}] n_clones = {len(coords_df)}")
display(coords_df.head())
fig

## 4. Task 4 — `energy_shift`

Requires a second real repertoire — edit `REPERTOIRE_2_PATH` above and rerun from the top if you haven't yet.

In [ ]:
df2 = load_repertoire(REPERTOIRE_2_PATH)

t0 = time.time()
per_gene_energy, weighted_summary = tm.energy_shift(df1, df2, downsample=DOWNSAMPLE, seed=0, device="auto")
print(f"energy_shift took {time.time() - t0:.1f}s")

print("\nPer-V-gene energy distance:")
display(per_gene_energy.sort_values("energy_distance", ascending=False).head(10))

print("\nFrequency-weighted summary:")
display(weighted_summary)

## 5. `embedding_type` option — sanity check

`embed_repertoire` can return either the raw 480-dim "pretrained" (local-structure) embedding, or the
default 64-dim "final" (contrastively fine-tuned, overall-structure) embedding. `compute_css` / `plot_umap` /
`energy_shift` / `build_reference` always use `embedding_type="final"` internally regardless of what's passed
to them, since TCRmeta's own repertoire-level statistics (and the shipped reference map) are defined against
that 64-dim space. This cell checks both halves of that behavior.

In [ ]:
t0 = time.time()
emb_pretrained = tm.embed_repertoire(df1, device="auto", embedding_type="pretrained")
print(f"[pretrained] embedded {len(emb_pretrained)} clones in {time.time() - t0:.1f}s")

t0 = time.time()
emb_final = tm.embed_repertoire(df1, device="auto", embedding_type="final")
print(f"[final]      embedded {len(emb_final)} clones in {time.time() - t0:.1f}s")

example_key = next(iter(emb_pretrained))
pretrained_dim = emb_pretrained[example_key].shape[0]
final_dim = emb_final[example_key].shape[0]
print("pretrained embedding dim:", pretrained_dim, "(expect 480)")
print("final embedding dim:     ", final_dim, "(expect 64)")

assert pretrained_dim == 480, f"expected 480-dim pretrained embedding, got {pretrained_dim}"
assert final_dim == 64, f"expected 64-dim final embedding, got {final_dim}"
assert not np.allclose(emb_pretrained[example_key][:64], emb_final[example_key]), (
    "pretrained and final embeddings for the same clone should not be identical"
)
print("OK: embed_repertoire's embedding_type option returns the expected, distinct embeddings.")

In [ ]:
# compute_css / plot_umap / energy_shift must IGNORE embedding_type and always score in the
# 64-dim "final" space (matching the reference map) — passing embedding_type="pretrained" here should
# NOT raise a dimension-mismatch error, and should give the same result as not passing it at all.
per_gene_css_ignored, whole_css_ignored = tm.compute_css(
    df1,
    reference=reference,
    downsample=DOWNSAMPLE,
    seed=SEED,
    device="auto",
    embedding_type="pretrained",  # should be silently overridden to "final" internally
)

assert np.isclose(whole_css_ignored.loc[0, "CSS"], whole_css.loc[0, "CSS"]), (
    "compute_css's result changed when embedding_type was passed — it should always force 'final' internally and ignore this kwarg"
)
print("OK: compute_css ignores embedding_type and always scores against the 64-dim reference space.")

## Notes

- If this notebook's `n_clones`/figure for `V_GENE` still doesn't match `debug_plot_umap.ipynb`, double check both
  notebooks point at the literal same `SAMPLE_CSV`/`MODEL_PATH` files, and that `DOWNSAMPLE`/`SEED`/`GRID_N` are identical
  (they should be here by construction — 10000 / 42 / 300 — matching `debug_plot_umap.ipynb`'s `TARGET_DEPTH`/`SEED`/`GRID_N`).
- `embed_repertoire`'s output is reused (`embeddings=embeddings`) for both `compute_css` and `plot_umap` so embedding
  computation only happens once and can't itself be a source of difference between the two calls.